## Install AlphaGenome  
https://github.com/google-deepmind/alphagenome_research/tree/main

In [ ]:

# !git clone https://github.com/google-deepmind/alphagenome_research.git
# !pip install -e ./alphagenome_research

In [10]:
# prepare a exons-only gtf feature file

import pyranges as pr


gtf_path = "data/annotations/gencode.v46.annotation.gtf"
out_path = "data/annotations/gencode.v46.annotation.plus_strand.gtf.feather"

gtf_df = pr.read_gtf(
    gtf_path,
    as_df=True,
    duplicate_attr=True
)

filtered_gtf = gtf_df[
    gtf_df["Strand"] == "+"
].copy()

filtered_gtf.to_feather(out_path)

print(filtered_gtf["Feature"].value_counts())

'''
gtf_path = "data/annotations/gencode.v46.annotation.gtf"       
out_path = "data/annotations/gencode.v46.annotation_exons.gtf.feather"

gtf_df = pr.read_gtf(
    gtf_path,
    as_df=True,
    duplicate_attr=True
)

custom_gtf = gtf_df[
    (gtf_df["Feature"] == "exon") &
    (gtf_df["gene_type"] == "protein_coding")
].copy()

custom_gtf.to_feather(out_path)
'''

Feature
exon              850171
CDS               457116
UTR               196199
transcript        130402
start_codon        50332
stop_codon         47494
gene               32001
Selenocysteine        53
Name: count, dtype: int64


'\ngtf_path = "data/annotations/gencode.v46.annotation.gtf"       \nout_path = "data/annotations/gencode.v46.annotation_exons.gtf.feather"\n\ngtf_df = pr.read_gtf(\n    gtf_path,\n    as_df=True,\n    duplicate_attr=True\n)\n\ncustom_gtf = gtf_df[\n    (gtf_df["Feature"] == "exon") &\n    (gtf_df["gene_type"] == "protein_coding")\n].copy()\n\ncustom_gtf.to_feather(out_path)\n'

In [1]:
import os
import jax

os.environ["JAX_PLATFORM_NAME"] = "gpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(jax.devices())
print("default backend:", jax.default_backend())


[CudaDevice(id=0)]
default backend: gpu


In [2]:
from alphagenome.data import genome
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models import variant_scorers
from alphagenome.models import dna_client

import matplotlib.pyplot as plt
import functools
from typing import Callable
from tqdm import tqdm
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn import metrics
from sklearn.metrics import average_precision_score, roc_auc_score

I0000 00:00:1777977735.406308  749914 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777977738.922736  749914 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
# Flags to improve determinism.
os.environ['XLA_FLAGS'] = ' '.join([
    '--xla_gpu_deterministic_ops',
    '--xla_gpu_enable_scatter_determinism_expander=True',
    '--xla_gpu_enable_triton_gemm=False',
])
# Increase GPU and CPU memory to reduce out of memory errors.
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'

```python
 dna_model.Organism.HOMO_SAPIENS: OrganismSettings(
          fasta_path=(
              'https://storage.googleapis.com/alphagenome/reference/gencode/'
              'hg38/GRCh38.p13.genome.fa'
          ),
          gtf_feather_path=(
              'https://storage.googleapis.com/alphagenome/reference/gencode/'
              'hg38/gencode.v46.annotation.gtf.gz.feather'
          ),
          pas_feather_path=(
              'https://storage.googleapis.com/alphagenome/reference/exon/hg38/'
              'polyadb_human_v3_exon3_contiguous_gtfv46.feather'
          ),
          splice_site_starts_feather_path=(
              'https://storage.googleapis.com/alphagenome/reference/gencode/'
              'hg38/gencode.v46.splice_sites_starts.feather'
          ),
          splice_site_ends_feather_path=(
              'https://storage.googleapis.com/alphagenome/reference/gencode/'
              'hg38/gencode.v46.splice_sites_ends.feather'
          ),
      ),
```

In [11]:
custom_settings = {
    dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings(
        fasta_path='https://storage.googleapis.com/alphagenome/reference/gencode/'
              'hg38/GRCh38.p13.genome.fa',
        gtf_feather_path="data/annotations/gencode.v46.annotation.plus_strand.gtf.feather",
        #gtf_feather_path=(
        #      'https://storage.googleapis.com/alphagenome/reference/gencode/'
        #      'hg38/gencode.v46.annotation.gtf.gz.feather'
        # ),
        pas_feather_path=(
              'https://storage.googleapis.com/alphagenome/reference/exon/hg38/'
              'polyadb_human_v3_exon3_contiguous_gtfv46.feather'
          ),
        splice_site_starts_feather_path=(
            'https://storage.googleapis.com/alphagenome/reference/gencode/'
            'hg38/gencode.v46.splice_sites_starts.feather'
        ),
        splice_site_ends_feather_path=(
            'https://storage.googleapis.com/alphagenome/reference/gencode/'
            'hg38/gencode.v46.splice_sites_ends.feather'
        ),
    )
}



model = dna_model.create_from_huggingface(
    "all_folds",
    organism_settings=custom_settings,
)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/opt/modules/i12g/anaconda/envs/IDPproject_indel_26/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


In [ ]:
# load the model from huggingface
# need to be logged in to huggingface and have access to the model

# model = dna_model.create_from_huggingface('all_folds')

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/opt/modules/i12g/anaconda/envs/IDPproject_indel_26/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/opt/modules/i12g/anaconda/envs/IDPproject_indel_26/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


In [12]:
def score_variant(variant_id: str, variant_scorer: Callable) -> pd.DataFrame: 
    
    # variant_id: in the form of chr3_4491276_T_C_b38 or chr3_4491276_T_C
    # variant_scorer: variant_scorers.RECOMMENDED_VARIANT_SCORERS[]

    parts = variant_id.split("_")
    if len(parts) == 5:
        chrom, pos, ref, alt, ext = parts
    elif len(parts) == 4:
        chrom, pos, ref, alt = parts
        ext = None
    else:
        raise ValueError(f"Unexpected variant_id format: {variant_id}")

    # build variant object
    variant = genome.Variant(
        chromosome=chrom,
        position=int(pos),
        reference_bases=ref,
        alternate_bases=alt,
    )
    # Create a 1MB interval centered on the variant.
    interval = variant.reference_interval.resize(2**20)
    
    variant_scores = model.score_variant(
        interval=interval, 
        variant=variant, 
        variant_scorers=[variant_scorer],
        organism=dna_client.Organism.HOMO_SAPIENS,
    )
    variant_scores = variant_scorers.tidy_scores(variant_scores)
    
    return variant_scores

## Reproduction of Alphagenome eQTL results

In [13]:
# import variants from alphagenome source


urls = ["https://storage.googleapis.com/alphagenome/evals/eqtl_variant_borzoi_coefficient_human_predictions.feather",
        "https://storage.googleapis.com/alphagenome/evals/eqtl_variant_catalogue_causality_gene_balanced_human_predictions.feather"]

dfs = [pd.read_feather(url) for url in urls]

df_coefficient = dfs[0]
df_zeroshot = dfs[1]
df_coefficient = df_coefficient.dropna(subset=['target'])
df_zeroshot = df_zeroshot.dropna(subset=['target'])

df_coefficient.head()

,variant_id,gene_id,tissue,prediction,target,variant_scorer,output_type,metric_calculator,metric_name
0,chr3_4491276_T_C_b38,ENSG00000231249.3,Adipose_Subcutaneous,0.101731,0.279811,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None
1,chr3_8733693_G_T_b38,ENSG00000182533.7,Adipose_Subcutaneous,0.014747,0.567912,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None
2,chr3_8963418_C_T_b38,ENSG00000070950.10,Adipose_Subcutaneous,-0.095976,-0.684296,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None
3,chr3_9867065_G_C_b38,ENSG00000187288.12,Adipose_Subcutaneous,-0.000609,-0.130677,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None
4,chr3_9903375_C_T_b38,ENSG00000163701.19,Adipose_Subcutaneous,-0.035684,-0.660048,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None


In [14]:
unique_variants = df_coefficient["variant_id"].unique()[:100] # take 100 variants for example

variant_scorer = variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']  # scorer for RNA seq --> RNA abundance


In [15]:
variant_scores_dic = {}
for variant in tqdm(unique_variants):
    variant_scores_dic[variant] = score_variant(variant, variant_scorer)


100%|██████████| 100/100 [05:06<00:00,  3.07s/it]


In [ ]:
# first variant positive strand only
variant_scores_dic[unique_variants[0]] # score per gene per ontology (per gtex_tissue)

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,...,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,gtex_tissue,data_source,endedness,genetically_modified,raw_score
0,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000287720,ENSG00000287720,lncRNA,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,CL:0000047,neuronal stem cell,in_vitro_differentiated_cells,embryonic,,encode,paired,False,-0.000073
1,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000287720,ENSG00000287720,lncRNA,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,total RNA-seq,CL:0000062,osteoblast,primary_cell,adult,,encode,paired,False,0.000013
2,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000287720,ENSG00000287720,lncRNA,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,CL:0000084,T-cell,primary_cell,adult,,encode,paired,False,0.000079
3,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000287720,ENSG00000287720,lncRNA,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,total RNA-seq,CL:0000084,T-cell,primary_cell,adult,,encode,single,False,0.000078
4,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000287720,ENSG00000287720,lncRNA,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,total RNA-seq,CL:0000115,endothelial cell,in_vitro_differentiated_cells,adult,,encode,single,False,-0.000068
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018115,left renal pelvis,tissue,embryonic,,encode,single,False,0.000080
1976,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018116,right renal pelvis,tissue,embryonic,,encode,single,False,0.000075
1977,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018117,left renal cortex interstitium,tissue,embryonic,,encode,single,False,0.000064
1978,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018118,right renal cortex interstitium,tissue,embryonic,,encode,single,False,-0.000082


In [9]:
# first variant  exons only
variant_scores_dic[unique_variants[0]] # score per gene per ontology (per gtex_tissue)

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,...,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,gtex_tissue,data_source,endedness,genetically_modified,raw_score
0,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000144455,SUMF1,protein_coding,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,CL:0000047,neuronal stem cell,in_vitro_differentiated_cells,embryonic,,encode,paired,False,0.000536
1,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000144455,SUMF1,protein_coding,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,total RNA-seq,CL:0000062,osteoblast,primary_cell,adult,,encode,paired,False,0.003278
2,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000144455,SUMF1,protein_coding,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,CL:0000084,T-cell,primary_cell,adult,,encode,paired,False,0.002532
3,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000144455,SUMF1,protein_coding,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,total RNA-seq,CL:0000084,T-cell,primary_cell,adult,,encode,single,False,0.002035
4,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000144455,SUMF1,protein_coding,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,total RNA-seq,CL:0000115,endothelial cell,in_vitro_differentiated_cells,adult,,encode,single,False,0.001350
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5143,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018115,left renal pelvis,tissue,embryonic,,encode,single,False,0.000080
5144,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018116,right renal pelvis,tissue,embryonic,,encode,single,False,0.000075
5145,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018117,left renal cortex interstitium,tissue,embryonic,,encode,single,False,0.000064
5146,chr3:4491276:T>C,chr3:3966988-5015564:.,ENSG00000134107,BHLHE40,protein_coding,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,polyA plus RNA-seq,UBERON:0018118,right renal cortex interstitium,tissue,embryonic,,encode,single,False,-0.000082


For the reproduction the raw results need to be filtered according to the Alphagenome data

In [19]:
df_coefficient_results = []

for row_nr in range(len(df_coefficient)):
    row = df_coefficient.iloc[row_nr]
    variant = row.variant_id
    
    if variant not in variant_scores_dic:
        continue
    
    variant_scores = variant_scores_dic[variant]
    
    mask = (
        (variant_scores["gene_id"] == row.gene_id.split('.')[0]) &
        (variant_scores["gtex_tissue"] == row.tissue)
    )

    filtered = variant_scores.loc[mask, "raw_score"]

    new_prediction = filtered.iloc[0] if not filtered.empty else np.nan
    # add prediction from model to the row and then collect row in a new dataframe for all variants
    new_row = row.copy()
    new_row['prediction_new'] = new_prediction

    df_coefficient_results.append(new_row)


In [20]:
df_coefficient_results = pd.DataFrame(df_coefficient_results)
df_coefficient_results.head()

,variant_id,gene_id,tissue,prediction,target,variant_scorer,output_type,metric_calculator,metric_name,prediction_new
0,chr3_4491276_T_C_b38,ENSG00000231249.3,Adipose_Subcutaneous,0.101731,0.279811,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,0.101454
1,chr3_8733693_G_T_b38,ENSG00000182533.7,Adipose_Subcutaneous,0.014747,0.567912,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.002932
2,chr3_8963418_C_T_b38,ENSG00000070950.10,Adipose_Subcutaneous,-0.095976,-0.684296,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.093976
3,chr3_9867065_G_C_b38,ENSG00000187288.12,Adipose_Subcutaneous,-0.000609,-0.130677,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.006362
4,chr3_9903375_C_T_b38,ENSG00000163701.19,Adipose_Subcutaneous,-0.035684,-0.660048,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.021424


```python
params: hk.Params,
state: hk.State,
apply_fn: ApplyFn,
junctions_apply_fn: JunctionsApplyFn,
metadata: Mapping[dna_model.Organism, AlphaGenomeOutputMetadata],
fasta_extractors: (
    Mapping[dna_model.Organism, fasta.FastaExtractor] | None
) = None,
splice_site_extractors: (
    Mapping[dna_model.Organism, splicing_io.SpliceSiteAnnotationExtractor]
    | None
) = None,
gtfs: Mapping[dna_model.Organism, pd.DataFrame] | None = None,
pas_gtfs: Mapping[dna_model.Organism, pd.DataFrame] | None = None,
num_splice_sites: int = 512,
splice_site_threshold: float = 0.1,
device: jax.Device | None = None,
```